# NGS Amplicon Analysis Pipeline for CRISPR Genome Editing Assessment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Innocent250/ngs-analysis-pipeline/blob/main/NGS_Analysis_Pipeline.ipynb)

**Author:** Innocent Byiringiro | University of Maryland, College Park | [ibyiring@umd.edu](mailto:ibyiring@umd.edu)
<br>**Lab:** [Qi Lab - Plant Genome Engineering](https://www.yipingqi.com/)

---

## About This Pipeline

This notebook provides an end-to-end bioinformatics pipeline for analyzing CRISPR-Cas genome editing outcomes from **Illumina MiSeq amplicon sequencing** data generated using the **Hi-TOM** (High-Throughput Tracking of Mutations) barcoding system ([Liu et al., 2019](https://doi.org/10.1007/s11427-019-9570-x)).

### What It Does

| Step | Description | Tool |
|------|-------------|------|
| 1. **Read Merging** | Merge overlapping paired-end reads (R1 + R2) into full-length amplicons | flashpy or fastp |
| 2. **Demultiplexing** | Split merged reads by Hi-TOM barcode pairs into individual samples | Custom Python |
| 3. **Editing Quantification** | Quantify indel frequencies, allele distributions, and mutation profiles | CRISPResso2 |
| 4. **Export** | Package results for downstream visualization and interpretation | - |

### Pipeline Overview

```
  WET LAB                                         COMPUTATIONAL (this notebook)
 ──────────────────────────────────               ──────────────────────────────────────
  Genomic DNA extraction (CTAB)                    FASTQ R1 + R2 (from sequencing)
       │                                                │
       ▼                                                ▼
  PCR Round 1: Target amplification                Step 1: Merge paired-end reads
  (target primers + bridging sequences)            (flashpy or fastp)
       │                                                │
       ▼                                                ▼
  PCR Round 2: Hi-TOM barcoding                    Step 2: Demultiplex by barcodes
  (barcodes + Illumina adapters*)                  (split into per-sample FASTQs)
       │                                                │
       ▼                                                ▼
  Pool samples → PCR cleanup →                     Step 3: CRISPResso2 batch analysis
  Illumina MiSeq 2x250bp sequencing               (indels, allele frequencies)
  (e.g., Amplicon-EZ by Azenta)                         │
                                                        ▼
                                                   Step 4: Export & visualize results

  * Illumina adapter sequences are specific to the sequencing provider.
    This pipeline uses adapters for Azenta's Amplicon-EZ service.
```

### How to Run This Notebook

**Option A: Google Colab (Recommended -- no local setup needed)**
1. Click the **"Open in Colab"** badge above
2. In Colab: `Runtime` > `Change runtime type` > select **Python 3** (GPU not required)
3. Upload your paired-end `.fastq.gz` files (file browser on the left, or mount Google Drive)
4. Prepare a barcode CSV mapping sample names to barcode sequences (template: `barcodes/hi_tom_barcodes.csv`)
5. Run cells sequentially from top to bottom, updating file paths and parameters as indicated
6. Download the zipped CRISPResso2 output at the end

**Option B: Local Jupyter**
```bash
git clone https://github.com/Innocent250/ngs-analysis-pipeline.git
cd ngs-analysis-pipeline
pip install -r requirements.txt
jupyter notebook NGS_Analysis_Pipeline.ipynb
```

### Input Requirements

| Input | Format | Description |
|-------|--------|-------------|
| Paired-end reads | `.fastq` or `.fastq.gz` | Illumina MiSeq R1 and R2 files |
| Barcode table | `.csv` | Columns: `Sample`, `Barcode_L`, `Barcode_R` |
| Amplicon sequence | Text string | Full reference amplicon (including primer regions) |
| Guide RNA sequence | Text string | 20-nt guide sequence (without PAM) |

### Associated Publication

> Byiringiro, I.\*, Contiliani, D.F.\*, Davies, C., Creste, S., & Qi, Y. *Targeting A/T Rich PAM Sites for Advanced Genome Engineering in Plants by Expanded CRISPR-Combo Systems.* (In preparation)

---
## Step 1: Environment Setup

This cell installs all required dependencies:

- **[flashpy](https://github.com/ponnhide/flashpy)** by [@ponnhide](https://github.com/ponnhide) -- A Python/Cython wrapper around the FLASH algorithm for merging paired-end reads with quality-aware overlap detection.

> **Note:** CRISPResso2 (installed in Step 4) now uses **fastp** internally for read processing (replacing the deprecated FLASH dependency). No separate FLASH installation is needed.

In [ ]:
# ============================================================
# 1a. Set up working directory
#     All outputs will be saved under WORK_DIR.
#     On Colab this defaults to /content; change if running locally.
# ============================================================
import subprocess
import os
import sys
import gzip
import collections
from tqdm import tqdm
import pandas as pd

WORK_DIR = '/content'  #@param {type:"string"}
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

# ============================================================
# 1b. Install flashpy for read merging
#     Credit: flashpy by ponnhide (https://github.com/ponnhide/flashpy)
#     A Cython wrapper for the FLASH paired-end read merging algorithm
# ============================================================
flashpy_dir = os.path.join(WORK_DIR, 'flashpy')
if not os.path.exists(flashpy_dir):
    !git clone -q https://github.com/ponnhide/flashpy.git {flashpy_dir}
    subprocess.run(f'cd {flashpy_dir} && python setup.py build_ext --inplace',
                   shell=True, capture_output=True)
sys.path.insert(0, flashpy_dir)
import _flash as fl

%load_ext autoreload
%autoreload 2

# ============================================================
# (Optional) Mount Google Drive if your data is stored there
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')

print('Environment setup complete.')

---
## Step 2: Read Merging

Paired-end Illumina reads (R1 and R2) overlap in the middle of the amplicon. This step merges them into a single, full-length sequence per read pair.

### How it works

The `flash()` function (from [flashpy](https://github.com/ponnhide/flashpy) by [@ponnhide](https://github.com/ponnhide)) slides one read along the other to find the best-scoring overlap, then stitches the reads together. When the overlapping bases disagree, the base with the **higher Phred quality score** is kept.

**Key parameters you can adjust:**
| Parameter | Default | Description |
|-----------|---------|-------------|
| `min_overlap` | 10 bp | Minimum overlap length to accept a merge |
| `max_overlap` | 300 bp | Maximum overlap length to consider |
| `min_identity` | 0.1 | Minimum fraction of matching bases in the overlap |

> **Tip:** For Hi-TOM amplicons (typically 250-400 bp with 2x250 bp reads), the default parameters work well. If you see low merge rates, try decreasing `min_overlap` or `min_identity`.
>
> **Adjusting overlap range:** You can change `min_overlap` and `max_overlap` based on your amplicon size and read length. For shorter amplicons with large overlaps, increase `min_overlap` to reduce false merges. For longer amplicons with small overlaps, decrease `min_overlap` to recover more reads.

### 2a. Define merging functions

Run this cell to load the merging functions (no changes needed here).

In [ ]:
hamming = lambda x,y: sum(tuple(map(lambda a,b: 1 if a==b else 0, x, y))) / len(x)
convert_ascii = lambda x: [ord(asc) - 33 for asc in x]
def read_fastq(fastq_name):
    """Read fastq file"""
    seq_dict = collections.defaultdict(dict)
    if fastq_name.split(".")[-1] == "gz":
        f = gzip.open(fastq_name.replace("'","").replace("\\",""), mode="rt", encoding='utf-8')
    else:
        f = open(fastq_name.replace("'","").replace("\\",""))
    n = 0
    for line in f:
        if line[0] == "@" and n % 4 == 0:
            key = line[1:].rstrip()
            key = key.split(" ")[0]
            key = key.replace(":","_")
            seq_dict[key]["key"] = line[1:].rstrip()
        elif n%4 == 1:
            seq_dict[key]["seq"] = line.rstrip()
        elif n%4 == 2:
            seq_dict[key]["option"] = line.rstrip()
        elif n%4 == 3:
            seq_dict[key]["quality"] = [ord(asc) - 33 for asc in line.rstrip()]
        n += 1
    f.close()
    return seq_dict

def merge(seq1, seq2, score1, score2, min_overlap=10, max_overlap=300, allow_outies=False, min_identity=0.1, max_identity=1.0, cython=True):
    seq1 = seq1.upper()
    seq2 = seq2.upper()
    reverse = 0
    if len(seq1) >= len(seq2):
        seq2 = seq2.translate(str.maketrans("ATGCRYKMSWBDHV","TACGYRMKWSVHDB"))[::-1]
        score2 = score2[::-1]
    else:
        reverse = 1
        seq1 = seq1.translate(str.maketrans("ATGCRYKMSWBDHV","TACGYRMKWSVHDB"))[::-1]
        seq2, seq1 = seq1, seq2

    if cython == False:
        current_overlap  = 0
        current_score    = None
        current_identity = min_identity
        current_slide     = 0
        for i in range(len(seq1)+len(seq2)):
            slide = i
            if i < len(seq2) and allow_outies == True:
                overlap_length = i
                subseq1   = seq1[:overlap_length]
                subseq2   = seq2[-1*overlap_length:]
                subscore1 = score1[:overlap_length]
                subscore2 = score2[-1*overlap_length:]
            elif i < len(seq1):
                overlap_length = len(seq2)
                subseq1   = seq1[i-len(seq2):i]
                subseq2   = seq2
                subscore1 = score1[i-len(seq2):i]
                subscore2 = score2
            else:
                overlap_length = len(seq1) + len(seq2) - i
                subseq1   = seq1[-1*overlap_length:]
                subseq2   = seq2[:overlap_length:]
                subscore1 = score1[-1*overlap_length:]
                subscore2 = score2[:overlap_length]

            if min_overlap <= overlap_length <= max_overlap:
                identity = hamming(subseq1, subseq2)
                if identity > current_identity or (identity == current_identity and current_overlap == 0):
                    current_slide     = slide
                    current_identity  = identity
                    current_overlap   = overlap_length
                    current_subseq1   = subseq1
                    current_subseq2   = subseq2
                    current_subscore1 = subscore1
                    current_subscore2 = subscore2

                elif identity == current_identity:
                    n = 0
                    cscore_avg = 0
                    for cn1, cn2, cs1, cs2 in zip(current_subseq1, current_subseq2, current_subscore1, current_subscore2):
                        if cn1 != cn2:
                            cscore_avg += max(cs1, cs2)
                            n += 1
                    if n > 0:
                        cscore_avg = cscore_avg / n

                    n = 0
                    score_avg = 0
                    for n1, n2, s1, s2 in zip(subseq1, subseq2, subscore1, subscore2):
                        if n1 != n2:
                            score_avg += max(s1, s2)
                            n += 1
                    if n > 0:
                        score_avg = score_avg / n
                    if score_avg > cscore_avg:
                        current_slide     = slide
                        current_identity  = identity
                        current_overlap   = overlap_length
                        current_subseq1   = subseq1
                        current_subseq2   = subseq2
                        current_subscore1 = subscore1
                        current_subscore2 = subscore2
                else:
                    pass

                if identity >= max_identity:
                    break

        if current_identity < min_identity:
            return False
        else:
            overlap_seq   = ""
            overlap_score = []
            for cn1, cn2, cs1, cs2 in zip(current_subseq1, current_subseq2, current_subscore1, current_subscore2):
                if cs1 > cs2:
                    overlap_seq += cn1
                    overlap_score.append(cs1)
                else:
                    overlap_seq += cn2
                    overlap_score.append(cs2)

            if current_slide < len(seq2) and allow_outies == True:
                left_seq    = seq2[:-1*overlap_length]
                right_seq   = seq1[overlap_length:]
                left_score  = score2[:-1*overlap_length]
                right_score = score1[overlap_length:]
            else:
                left_seq    = seq1[:-1*overlap_length]
                right_seq   = seq2[overlap_length:]
                left_score  = score1[:-1*overlap_length]
                right_score = score2[overlap_length:]
            merged_seq   = left_seq + overlap_seq + right_seq
            merged_score = left_score + overlap_score + right_score
    else:
        merged_seq, merged_score, current_slide, current_overlap, current_identity = fl.merge(seq1.encode('utf-8'), seq2.encode('utf-8'), score1, score2, min_overlap, max_overlap, allow_outies, min_identity, max_identity)
    return merged_seq, merged_score, current_slide, current_overlap, current_identity

def flash(read1, read2, min_overlap=10, max_overlap=300, allow_outies=False, min_identity=0.1, max_identity=1.0, show_progress=True, key_check=True):
    r1_dict = read_fastq(read1)
    r2_dict = read_fastq(read2)

    dist_dict = collections.defaultdict(lambda:[0, 0])
    merged_dict = collections.defaultdict(dict)
    if show_progress == True:
        if key_check == True:
            keys = [key for key in r1_dict.keys() if key in r2_dict]
            keys = tqdm(keys)
        else:
            keys = tqdm(r1_dict.keys(), total=len(r1_dict))
    else:
        if key_check == True:
            keys = [key for key in r1_dict.keys() if key in r2_dict]
        else:
            keys = r1_dict.keys()

    for key in keys:
        seq1   = r1_dict[key]["seq"]
        seq2   = r2_dict[key]["seq"]
        score1 = r1_dict[key]["quality"]
        score2 = r2_dict[key]["quality"]
        result = merge(seq1, seq2, score1, score2, min_overlap, max_overlap, allow_outies, min_identity, max_identity)
        if result != False:
            merged_dict[key]["seq"]      = "".join(map(chr, result[0]))
            merged_dict[key]["quality"]  = result[1]
            merged_dict[key]["r1_key"]   = r1_dict[key]["key"]
            merged_dict[key]["r2_key"]   = r2_dict[key]["key"]
            merged_dict[key]["identity"] = result[4]
            if result[2] == 1:
                dist_dict["outie", result[3]][0] += 1
                dist_dict["outie", result[3]][1] += result[4]
            else:
                dist_dict["innie", result[3]][0] += 1
                dist_dict["innie", result[3]][1] += result[4]
    for key in dist_dict:
        dist_dict[key][1] = dist_dict[key][1] / dist_dict[key][0]

    return merged_dict, dist_dict

---
### 2b. Merge your paired-end reads

**Choose one of two options below:**

- **Option 1 (flashpy):** Uses the Python/Cython FLASH implementation loaded above. Good for most use cases.
- **Option 2 (fastp):** Uses [fastp](https://github.com/OpenGene/fastp) ([Chen et al., 2018](https://doi.org/10.1093/bioinformatics/bty560)), an ultrafast all-in-one FASTQ preprocessor that can merge reads, trim adapters, and generate QC reports in a single step.

**Instructions:**
1. Upload your R1 and R2 FASTQ files to the Colab environment (into `WORK_DIR`, or mount Google Drive)
2. Update the file names below (paths are built automatically from `WORK_DIR`)
3. Run **either** Option 1 or Option 2 (not both)

In [ ]:
# ============================================================
# OPTION 1: Merge with flashpy
# UPDATE the file names below. Place your FASTQ files in WORK_DIR
# or provide full paths.
# ============================================================
read_R1 = os.path.join(WORK_DIR, 'YOUR_SAMPLE_R1_001.fastq.gz')  #@param {type:"string"}
read_R2 = os.path.join(WORK_DIR, 'YOUR_SAMPLE_R2_001.fastq.gz')  #@param {type:"string"}
merged_fastq = os.path.join(WORK_DIR, 'YOUR_SAMPLE_merged.fastq') #@param {type:"string"}

# Merge paired-end reads using flashpy
merged_dict, dist_dict = flash(read_R1, read_R2)

# Print merge statistics
print(f'Total merged reads: {len(merged_dict):,}')
print(f'\nOverlap distribution (top 10 by count):')
sorted_dist = sorted(dist_dict.items(), key=lambda x: x[1][0], reverse=True)[:10]
for key, val in sorted_dist:
    print(f'  {key[0]} overlap={key[1]}bp: count={val[0]:,}, avg_identity={val[1]:.3f}')

# Write merged reads to FASTQ (proper 4-line format)
n = 0
with open(merged_fastq, 'w') as f:
    for key in merged_dict:
        seq = merged_dict[key]['seq']
        qual = ''.join(chr(q + 33) for q in merged_dict[key]['quality'])
        f.write(f"@{key}\n{seq}\n+\n{qual}\n")
        n += 1
print(f'\nWrote {n:,} merged reads to {merged_fastq}')

In [ ]:
# ============================================================
# OPTION 2: Merge with fastp (alternative to flashpy)
# Uncomment ALL lines below and run this cell INSTEAD of Option 1
# fastp also performs adapter trimming and quality filtering
# ============================================================

# # First, install fastp (only need to run once per session)
# !wget -q http://opengene.org/fastp/fastp && chmod a+x ./fastp && mv fastp /usr/local/bin/

# read_R1 = os.path.join(WORK_DIR, 'YOUR_SAMPLE_R1_001.fastq.gz')  #@param {type:"string"}
# read_R2 = os.path.join(WORK_DIR, 'YOUR_SAMPLE_R2_001.fastq.gz')  #@param {type:"string"}
# merged_fastq = os.path.join(WORK_DIR, 'YOUR_SAMPLE_merged.fastq') #@param {type:"string"}

# !fastp \
#     --in1 {read_R1} \
#     --in2 {read_R2} \
#     --merged_out {merged_fastq} \
#     --out1 /dev/null --out2 /dev/null \
#     --overlap_len_require 10 \
#     --overlap_diff_percent_limit 20 \
#     --merge \
#     --html {os.path.join(WORK_DIR, 'fastp_report.html')} \
#     --json {os.path.join(WORK_DIR, 'fastp_report.json')}

# print(f'Merging complete. Output: {merged_fastq}')
# print('QC report: fastp_report.html (download and open in browser)')

---
## Step 3: Demultiplex by Hi-TOM Barcodes

After merging, each read contains the full amplicon flanked by Hi-TOM barcodes. This step splits the merged reads into **individual sample FASTQ files** based on the barcode pairs assigned to each sample during PCR Round 2.

### How the Hi-TOM barcoding system works

In the Hi-TOM system ([Liu et al., 2019](https://doi.org/10.1007/s11427-019-9570-x)), each sample is assigned a unique combination of forward (F) and reverse (R) barcodes during the second round of PCR. With 12 forward barcodes (plate columns) and 8 reverse barcodes (plate rows), **96 unique sample barcodes** are possible per sequencing run.

```
Read structure after merging:
5'- [F barcode] - [bridging seq] - [TARGET AMPLICON] - [bridging seq] - [R barcode RC] -3'
```

The demultiplexer checks both orientations (forward-reverse and reverse-forward) to capture reads regardless of which strand was sequenced.

### 3a. Define demultiplexing function

Run this cell to load the function (no changes needed).

In [ ]:
def reverse_complement(dna):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join([complement.get(base, 'N') for base in dna[::-1]])

def split_fastq(df, fastq, output_directory):
    """
    Demultiplex a merged FASTQ file by Hi-TOM barcode pairs.

    Reads the FASTQ once, checks each read against all barcode pairs
    using exact matching, and writes matching reads to per-sample files.

    Parameters
    ----------
    df : DataFrame with columns 'Sample', 'Barcode_L', 'Barcode_R'
    fastq : path to merged FASTQ file (4-line format)
    output_directory : base directory for per-sample output folders
    """
    # Pre-compute barcode lookup
    barcodes = []
    for _, row in df.iterrows():
        sample = row['Sample']
        left = row['Barcode_L'].upper()
        right_raw = row['Barcode_R'].upper()
        right_rc = reverse_complement(right_raw)
        left_rc = reverse_complement(left)
        barcodes.append({
            'sample': sample,
            'left': left,
            'right_rc': right_rc,
            'right_raw': right_raw,
            'left_rc': left_rc,
            'left_len': len(left),
            'right_len': len(right_raw),
        })

    # Create output directories and open file handles
    file_handles = {}
    for bc in barcodes:
        sample = bc['sample']
        out_dir = os.path.join(output_directory, sample)
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, f'{sample}.extendedFrags.fastq')
        file_handles[sample] = open(out_path, 'w')

    # Read FASTQ once and assign each read
    stats = {bc['sample']: 0 for bc in barcodes}
    total_reads = 0
    unassigned = 0

    with open(fastq, 'r') as f:
        while True:
            header = f.readline().rstrip()
            if not header:
                break
            seq = f.readline().rstrip()
            plus = f.readline().rstrip()
            qual = f.readline().rstrip()
            total_reads += 1

            assigned = False
            for bc in barcodes:
                ll = bc['left_len']
                rl = bc['right_len']

                # Forward orientation: [F barcode]...[RC of R barcode]
                if len(seq) >= ll + rl:
                    if seq[:ll] == bc['left'] and seq[-rl:] == bc['right_rc']:
                        trimmed_seq = seq[ll:-rl]
                        trimmed_qual = qual[ll:-rl]
                        file_handles[bc['sample']].write(
                            f"{header}\n{trimmed_seq}\n+\n{trimmed_qual}\n")
                        stats[bc['sample']] += 1
                        assigned = True
                        break

                # Reverse orientation: [R barcode]...[RC of F barcode]
                if not assigned and len(seq) >= rl + ll:
                    if seq[:rl] == bc['right_raw'] and seq[-ll:] == bc['left_rc']:
                        trimmed_seq = seq[rl:-ll]
                        trimmed_qual = qual[rl:-ll]
                        file_handles[bc['sample']].write(
                            f"{header}\n{trimmed_seq}\n+\n{trimmed_qual}\n")
                        stats[bc['sample']] += 1
                        assigned = True
                        break

            if not assigned:
                unassigned += 1

    # Close all file handles
    for fh in file_handles.values():
        fh.close()

    # Print demultiplexing summary
    print(f'\n{"="*50}')
    print(f'Demultiplexing Summary')
    print(f'{"="*50}')
    print(f'Total reads:      {total_reads:,}')
    print(f'Assigned:         {total_reads - unassigned:,} ({100*(total_reads-unassigned)/max(total_reads,1):.1f}%)')
    print(f'Unassigned:       {unassigned:,} ({100*unassigned/max(total_reads,1):.1f}%)')
    print(f'\nPer-sample counts:')
    for bc in barcodes:
        s = bc['sample']
        print(f'  {s:30s}  {stats[s]:>8,} reads')
    return stats

### 3b. Run Demultiplexing

**Prepare your barcode CSV file** with the following columns:

| Column | Description | Example |
|--------|-------------|---------|
| `Sample` | Your sample name | `Plant_1` |
| `Barcode_L` | Forward (left) barcode sequence | `gcttGCGTt` |
| `Barcode_R` | Reverse (right) barcode sequence | `ctgtGCGTt` |

> **Template:** A complete 96-well plate barcode CSV is available at `barcodes/hi_tom_barcodes.csv` in this repository. Copy it, replace the `Sample` column with your sample names, and remove unused wells.

**Update the file names below** (paths are built automatically from `WORK_DIR`):

In [ ]:
# ============================================================
# UPDATE THESE for your experiment
# ============================================================
barcode_csv = os.path.join(WORK_DIR, 'YOUR_BARCODE_TABLE.csv')  #@param {type:"string"}
# merged_fastq was defined in Step 2 above

demux_output = os.path.join(WORK_DIR, 'YOUR_SAMPLE_Split')  #@param {type:"string"}

reference_df = pd.read_csv(barcode_csv)

# Run demultiplexing (reads file once, exact barcode matching)
demux_stats = split_fastq(
    df=reference_df,
    fastq=merged_fastq,
    output_directory=demux_output
)

---
## Step 4: Install CRISPResso2

[CRISPResso2](https://github.com/pinellolab/CRISPResso2) ([Clement et al., 2019](https://doi.org/10.1038/s41587-019-0032-3)) quantifies genome editing outcomes from deep sequencing data:

- **Indel frequencies** (insertions and deletions around the cut site)
- **Allele frequency tables** (every unique sequence and its count)
- **Mutation profiles** (position-specific editing rates)
- **Publication-ready plots** (HTML and PNG)

> **Note:** CRISPResso2 v2.3+ uses **fastp** internally for read processing (replacing the previously required FLASH). No separate FLASH installation is needed.
>
> Installation takes 2-3 minutes. You only need to run this once per Colab session.

In [ ]:
# Install CRISPResso2 (v2.3+ uses fastp internally, no FLASH needed)
!pip install -q CRISPResso2

---
## Step 5: Prepare CRISPResso2 Batch File

This step generates a **batch settings file** that tells CRISPResso2 where each sample's FASTQ file is and what amplicon/guide to use.

**Update the parameters below:**

| Parameter | Description | Example |
|-----------|-------------|---------|
| `project_directory` | Path to your demultiplexed output folder from Step 3 | Auto-filled from Step 3 |
| `amplicon_seq` | Full reference amplicon sequence (the expected WT sequence) | `TCAGAGAG...CCTC` |
| `guide_seq` | 20-nt guide RNA sequence, **without PAM** | `TGCCTTCCATCGTCAGCACA` |

> **Important:** The `amplicon_seq` should be the complete expected sequence of your PCR product (after barcode trimming). The `guide_seq` should match exactly 20 nt of the amplicon where Cas9/Cas12 cuts.

In [ ]:
# ============================================================
# UPDATE THESE PARAMETERS for your experiment
# ============================================================
project_directory = demux_output  # From Step 3 above
amplicon_seq = 'YOUR_AMPLICON_SEQUENCE_HERE'              #@param {type:"string"}
guide_seq = 'YOUR_GUIDE_RNA_20NT'                         #@param {type:"string"}
batch_settings_file = os.path.join(WORK_DIR, 'YOUR_SAMPLE_batch.tsv')  #@param {type:"string"}

# Build batch settings: CRISPResso2 needs 'name' and 'fastq_r1' per sample.
# amplicon_seq and guide_seq are passed as command-line args in Step 6.
batch_settings = pd.DataFrame(columns=['name', 'fastq_r1'])

ignore_dirs = ['.ipynb_checkpoints']

for sample_name in os.listdir(project_directory):
    if sample_name in ignore_dirs:
        continue

    sample_path = os.path.join(project_directory, sample_name)
    if os.path.isdir(sample_path):
        fastq_files = [f for f in os.listdir(sample_path) if f.endswith('.fastq')]
        if not fastq_files:
            print(f'No .fastq files found in {sample_path}. Skipping...')
            continue
        fastq_r1 = os.path.join(sample_path, fastq_files[0])

        new_row = pd.DataFrame({
            'name': [sample_name],
            'fastq_r1': [fastq_r1],
        })
        batch_settings = pd.concat([batch_settings, new_row], ignore_index=True)

batch_settings.to_csv(batch_settings_file, sep='\t', index=False)
print(f'Batch file created with {len(batch_settings)} samples: {batch_settings_file}')
batch_settings.head()

---
## Step 6: Run CRISPResso2 Batch Analysis

Execute CRISPResso2 in batch mode across all demultiplexed samples. This is the core analysis step.

**Full documentation:** [CRISPResso2 GitHub](https://github.com/pinellolab/CRISPResso2) | Run `!CRISPResso -h` in a cell to see all available parameters.

### Editing mode

Uncomment **only one** of the three command blocks below depending on your experiment type:

- **Genome editing (Cas nuclease):** Quantifies indels (insertions/deletions) at the cut site. This is the default.
- **Base editing (CBE/ABE):** Quantifies C-to-T or A-to-G conversions in the editing window. Requires `--base_editor_output`.
- **Prime editing:** Quantifies precise edits against an expected edited sequence. Requires `--prime_editing_pegRNA_extension_quantification_window_size`.

### Key parameters reference

| Parameter | Flag | Default | Description |
|-----------|------|---------|-------------|
| Amplicon sequence | `--amplicon_seq` | *required* | WT reference amplicon sequence |
| Guide RNA | `-g` | *required* | 20-nt guide sequence (without PAM) |
| Window center | `-wc` | -3 | Center of quantification window relative to the cut site (use -10 for Cas12) |
| Window size | `-w` | 1 | Size of the quantification window (bp); increase for broader analysis |
| Min. alignment score | `--min_average_read_quality` | 0 | Minimum average quality to keep a read |
| Ignore substitutions | `--ignore_substitutions` | False | If set, only count insertions/deletions as modifications (recommended for genome editing to get pure indel frequency) |
| Exclude bp from sides | `--exclude_bp_from_left` / `--exclude_bp_from_right` | 15 | Exclude N bp from each end to avoid primer artifacts |
| Base editor output | `--base_editor_output` | False | Enable base editing quantification mode |
| Prime edit pegRNA | `--prime_editing_pegRNA_extension_quantification_window_size` | 0 | Window for prime editing quantification |

> **Tip for genome editing:** Add `--ignore_substitutions` so that the reported modification percentage reflects **indel frequency only**, excluding any PCR/sequencing-introduced substitutions.

In [ ]:
# ============================================================
# CRISPResso2 parameters -- UPDATE as needed
# Full docs: https://github.com/pinellolab/CRISPResso2
# Run !CRISPResso -h to see all available parameters
# ============================================================
batch_file = batch_settings_file  # From Step 5
window_center = -10  #@param {type:"integer"}
window_size = 20     #@param {type:"integer"}

# ============================================================
# GENOME EDITING (default) -- Cas nuclease indel quantification
# Uncomment the command below for standard genome editing analysis.
# --ignore_substitutions ensures modification % = indel frequency only.
# ============================================================
!CRISPRessoBatch \
    --batch_settings {batch_file} \
    --amplicon_seq {amplicon_seq} \
    -g {guide_seq} \
    -wc {window_center} \
    -w {window_size} \
    --ignore_substitutions

# ============================================================
# BASE EDITING (CBE or ABE) -- Uncomment below INSTEAD of above
# Quantifies C>T (CBE) or A>G (ABE) conversions in the window.
# Docs: https://github.com/pinellolab/CRISPResso2#base-editing
# ============================================================
# !CRISPRessoBatch \
#     --batch_settings {batch_file} \
#     --amplicon_seq {amplicon_seq} \
#     -g {guide_seq} \
#     -wc {window_center} \
#     -w {window_size} \
#     --base_editor_output

# ============================================================
# PRIME EDITING -- Uncomment below INSTEAD of above
# Requires the expected edited sequence for comparison.
# Docs: https://github.com/pinellolab/CRISPResso2#prime-editing
# ============================================================
# expected_edited_seq = 'YOUR_EXPECTED_EDITED_AMPLICON'  #@param {type:"string"}
# !CRISPRessoBatch \
#     --batch_settings {batch_file} \
#     --amplicon_seq {amplicon_seq} \
#     -g {guide_seq} \
#     -wc {window_center} \
#     -w {window_size} \
#     --expected_hdr_amplicon_seq {expected_edited_seq} \
#     --prime_editing_pegRNA_extension_quantification_window_size 5

---
## Step 7: Export Results

Zip and download the CRISPResso2 output directory for downstream analysis and visualization.

In [ ]:
# Zip the CRISPResso2 output
import glob

# Find the CRISPRessoBatch output directory
batch_output = glob.glob(os.path.join(WORK_DIR, 'CRISPRessoBatch_on_*'))
if batch_output:
    output_dir = batch_output[0]
    zip_path = output_dir + '.zip'
    !zip -r {zip_path} {output_dir}
    print(f'Zipped: {zip_path}')

    # Download (Colab only)
    from google.colab import files
    files.download(zip_path)
else:
    print('No CRISPRessoBatch output found. Check the previous step for errors.')

---
## Step 8: (Optional) R / ggplot2 Visualization in Colab

If you prefer to create publication-quality figures using **R and ggplot2** directly in this notebook, you can use the `rpy2` integration below. This is entirely optional -- you can also download the CRISPResso2 output files and plot them in RStudio.

> **Note:** The cells below are commented out by default. Uncomment and run them if you want to use R within this Colab session.

In [ ]:
# ============================================================
# 8a. Set up R environment in Colab (uncomment all lines to use)
# ============================================================

# # Load rpy2 for R integration
# %load_ext rpy2.ipython

# # Install R packages (first run only -- takes a few minutes)
# %%R
# if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
# if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2", repos = "https://cloud.r-project.org")
# if (!requireNamespace("readr", quietly = TRUE)) install.packages("readr", repos = "https://cloud.r-project.org")
# if (!requireNamespace("dplyr", quietly = TRUE)) install.packages("dplyr", repos = "https://cloud.r-project.org")
# if (!requireNamespace("tidyr", quietly = TRUE)) install.packages("tidyr", repos = "https://cloud.r-project.org")
# if (!requireNamespace("scales", quietly = TRUE)) install.packages("scales", repos = "https://cloud.r-project.org")

In [ ]:
# ============================================================
# 8b. Import CRISPResso2 results into R and create a basic plot
#     Uncomment all lines below to use.
#     Adjust the path to point to your CRISPRessoBatch output.
# ============================================================

# %%R
# library(tidyverse)
#
# # ------------------------------------------------------------------
# # Import CRISPResso2 quantification results
# # Update the path below to your CRISPRessoBatch output directory
# # ------------------------------------------------------------------
# batch_dir <- "CRISPRessoBatch_on_YOUR_SAMPLE_batch"
#
# # Collect editing frequency from each sample subfolder
# sample_dirs <- list.dirs(batch_dir, recursive = FALSE)
# results <- data.frame()
#
# for (d in sample_dirs) {
#   quant_file <- file.path(d, "CRISPResso_quantification_of_editing_frequency.txt")
#   if (file.exists(quant_file)) {
#     df <- read.delim(quant_file, header = TRUE)
#     df$sample <- basename(d)
#     results <- bind_rows(results, df)
#   }
# }
#
# # ------------------------------------------------------------------
# # Basic bar plot of editing efficiency per sample
# # ------------------------------------------------------------------
# if (nrow(results) > 0 && "Reads_aligned_..Modification" %in% colnames(results)) {
#   ggplot(results, aes(x = reorder(sample, -Reads_aligned_..Modification),
#                        y = Reads_aligned_..Modification)) +
#     geom_bar(stat = "identity", fill = "steelblue") +
#     theme_minimal(base_size = 12) +
#     theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
#     labs(x = "Sample", y = "Modification (%)",
#          title = "CRISPResso2: Editing Efficiency per Sample") +
#     scale_y_continuous(limits = c(0, 100))
# } else {
#   cat("No results found. Check your batch_dir path above.\n")
# }

---
## Step 9: Interpreting Results & Downstream Visualization

### What CRISPResso2 outputs

The batch output directory (`CRISPRessoBatch_on_*`) contains a folder for each sample with:

| File | Description |
|------|-------------|
| `CRISPResso_quantification_of_editing_frequency.txt` | Overall editing rates (% modified, unmodified) |
| `Alleles_frequency_table.zip` | Detailed allele frequency table (every unique sequence and its count) |
| `CRISPResso_mapping_statistics.txt` | Read mapping and quality metrics |
| `*.html` / `*.png` | Publication-ready plots (indel size distribution, allele plots, etc.) |

### Classifying T0 plant genotypes

For CRISPR-edited T0 plants, we classify editing outcomes based on indel frequency thresholds:

| Genotype | Indel Frequency | Interpretation |
|----------|----------------|----------------|
| Wild-type (WT) | < 10% | No editing detected |
| Chimeric | 10% - 30% | Somatic editing (mixed cell populations) |
| Monoallelic | 30% - 70% | One allele edited |
| Biallelic | > 70% | Both alleles edited |

### Visualization options

1. **CRISPResso2 built-in figures:** Open the HTML files in a browser for interactive plots
2. **Python (matplotlib/seaborn):** Import the `.txt` summary files into pandas for custom plotting
3. **R (ggplot2):** Use the optional R cells in Step 8 above, or export the summary tables and use R/RStudio locally

---

## Acknowledgments

- **flashpy** read merging: [ponnhide](https://github.com/ponnhide/flashpy)
- **CRISPResso2:** [Pinello Lab](https://github.com/pinellolab/CRISPResso2) ([Clement et al., 2019](https://doi.org/10.1038/s41587-019-0032-3))
- **Hi-TOM barcoding system:** [Liu et al., 2019](https://doi.org/10.1007/s11427-019-9570-x)
- **fastp:** [Chen et al., 2018](https://doi.org/10.1093/bioinformatics/bty560)

---

**Contact:** Innocent Byiringiro ([ibyiring@umd.edu](mailto:ibyiring@umd.edu)) | [Qi Lab](https://www.yipingqi.com/), University of Maryland, College Park